# Selecao Espectral YPNB + FWHM


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import find_peaks, peak_widths

In [ ]:
VIP_THRESHOLD=1.0
SPEARMAN_THRESHOLD=0.60
PEAK_PROMINENCE=0.02
YPNB_BUFFER_NM=30

In [ ]:
def calculate_fwhm_real_axis(signal, peaks, wavelengths):
    result=peak_widths(signal, peaks, rel_height=0.5)
    left_ips,right_ips=result[2],result[3]
    x=np.arange(len(wavelengths))
    left_nm=np.interp(left_ips,x,wavelengths)
    right_nm=np.interp(right_ips,x,wavelengths)
    return left_nm,right_nm,right_nm-left_nm

def overlap_interval(a_left,a_right,b_left,b_right):
    return not(a_right<b_left or a_left>b_right)

Defina wavelengths, vip_scores e spearman_corr antes de executar.

In [ ]:
corr_abs=np.abs(spearman_corr)
valid_idx=np.where((vip_scores>VIP_THRESHOLD)&(corr_abs>SPEARMAN_THRESHOLD))[0]
corr_filtered=np.zeros_like(corr_abs)
corr_filtered[valid_idx]=corr_abs[valid_idx]
peaks,_=find_peaks(corr_filtered,prominence=PEAK_PROMINENCE)
left_nm,right_nm,fwhm_nm=calculate_fwhm_real_axis(corr_filtered,peaks,wavelengths)

df_fwhm=pd.DataFrame({
"peak_wavelength":wavelengths[peaks],
"left_nm":left_nm,
"right_nm":right_nm,
"fwhm_nm":fwhm_nm,
"spearman":corr_abs[peaks],
"vip":vip_scores[peaks]}).sort_values("spearman",ascending=False)

In [ ]:
# Metodo YPNB
selected=[]
for _,row in df_fwhm.iterrows():
    p=row["peak_wavelength"]
    if len(selected)==0 or all(abs(p-s)>YPNB_BUFFER_NM for s in selected):
        selected.append(p)
method_ypnb=df_fwhm[df_fwhm["peak_wavelength"].isin(selected)]
print(method_ypnb.head())

In [ ]:
fig,ax=plt.subplots(2,1,figsize=(12,8),sharex=True)
ax[0].plot(wavelengths,corr_abs)
ax[0].scatter(df_fwhm["peak_wavelength"],df_fwhm["spearman"],c="r")
ax[0].set_title("Peaks")
ax[1].plot(wavelengths,corr_abs)
ax[1].scatter(method_ypnb["peak_wavelength"],method_ypnb["spearman"],c="g")
ax[1].set_title("YPNB")
plt.show()